In [0]:
"""
id: python_3
template: python
templateVersion: 1.0.0
name: API_Customers
position:
  x: 300
  y: 232.5
description:
  text: Load data from input if available; otherwise, fetch and load customer data from a remote CSV file.
  hash: ac43059c
previewCodeHash: 8877d02296a8a15d
previewMode: full
config:
  code: |
    if inputs.get("data"):
        result = inputs["data"][0]
    else:
        import requests
        import pandas as pd
        from io import StringIO
        url="https://raw.githubusercontent.com/anshlambagit/Databricks_Lakeflow_Designer/refs/heads/main/customers.csv"

        response=requests.get(url)

        csv_data=response.content.decode('utf-8')

        df=pd.read_csv(StringIO(csv_data))

        result = spark.createDataFrame(df)
input: []
"""

# generated from the system
from typing import Dict, Any
from pyspark.sql import DataFrame

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    data = inputs.get("data", [] if True else None)
    result = data[0] if data else spark.createDataFrame([], "col: string")

    if inputs.get("data"):
        result = inputs["data"][0]
    else:
        import requests
        import pandas as pd
        from io import StringIO
        url="https://raw.githubusercontent.com/anshlambagit/Databricks_Lakeflow_Designer/refs/heads/main/customers.csv"

        response=requests.get(url)

        csv_data=response.content.decode('utf-8')

        df=pd.read_csv(StringIO(csv_data))

        result = spark.createDataFrame(df)

    return {"result": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {}
inputs = {}
out = run(config, inputs, spark)
ctx["python_3.result"] = out["result"]

In [0]:
"""
id: python_4
template: python
templateVersion: 1.0.0
name: API_Shipment
position:
  x: 600
  y: 310
description:
  text: Load data from an input or fetch a shipments CSV file.
  hash: 1d72b77e
previewCodeHash: 2c25886d5e1629a4
previewMode: "1000"
config:
  code: |
    if inputs.get("data"):
        result = inputs["data"][0]
    else:
        import requests
        import pandas as pd
        from io import StringIO
        url="https://raw.githubusercontent.com/anshlambagit/Databricks_Lakeflow_Designer/refs/heads/main/shipments.csv"

        response=requests.get(url)

        csv_data=response.content.decode('utf-8')

        df=pd.read_csv(StringIO(csv_data))

        result = spark.createDataFrame(df)
input: []
"""

# generated from the system
from typing import Dict, Any
from pyspark.sql import DataFrame

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    data = inputs.get("data", [] if True else None)
    result = data[0] if data else spark.createDataFrame([], "col: string")

    if inputs.get("data"):
        result = inputs["data"][0]
    else:
        import requests
        import pandas as pd
        from io import StringIO
        url="https://raw.githubusercontent.com/anshlambagit/Databricks_Lakeflow_Designer/refs/heads/main/shipments.csv"

        response=requests.get(url)

        csv_data=response.content.decode('utf-8')

        df=pd.read_csv(StringIO(csv_data))

        result = spark.createDataFrame(df)

    return {"result": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {}
inputs = {}
out = run(config, inputs, spark)
ctx["python_4.result"] = out["result"]

In [0]:
"""
id: source_0
template: source
templateVersion: 2.0.0
name: orders
position:
  x: 0
  y: 0
description:
  text: Retrieve all data from the orders table.
  hash: "51051346"
previewCodeHash: aa59f2871d37258a
previewMode: "1000"
config:
  table_source:
    tableName: designer_catalog.raw.orders
input: []
"""

# generated from the system
from typing import Dict, Any, List

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "designer_catalog.raw.orders"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["source_0.data"] = out["data"]

In [0]:
"""
id: source_1
template: source
templateVersion: 2.0.0
name: order_items
position:
  x: 0
  y: 155
description:
  text: Load all data from the order_items table.
  hash: d5b7f701
previewCodeHash: 738433c5121ea367
previewMode: "1000"
config:
  table_source:
    tableName: designer_catalog.raw.order_items
input: []
"""

# generated from the system
from typing import Dict, Any, List

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "designer_catalog.raw.order_items"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["source_1.data"] = out["data"]

In [0]:
"""
id: source_14
template: source
templateVersion: 2.0.0
name: reviews
position:
  x: 0
  y: 465
description:
  text: Load all data from the reviews table.
  hash: bac52737
previewCodeHash: 2217a0d897c6b16d
previewMode: "1000"
config:
  table_source:
    tableName: designer_catalog.raw.reviews
input: []
"""

# generated from the system
from typing import Dict, Any, List

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "designer_catalog.raw.reviews"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["source_14.data"] = out["data"]

In [0]:
"""
id: join_2
template: join
templateVersion: 1.0.0
name: OrdersjoinOrders_items
position:
  x: 300
  y: 77.5
description:
  text: Left join on order_id and select specific order and order item details.
  hash: 9457e8db
previewCodeHash: 5e1d9e57a3a57aa9
previewMode: "1000"
config:
  join_type: left
  join_conditions: left.order_id = right.order_id
  expressions:
    - "`left`.order_id"
    - "`left`.customer_id"
    - "`left`.order_date"
    - "`left`.order_status"
    - "`right`.order_item_id"
    - "`right`.product_id"
    - "`right`.quantity"
    - "`right`.unit_price"
    - "`right`.discount_pct"
    - "`right`.line_total"
input:
  - node: source_0
    input_port: left
    output_port: data
  - node: source_1
    input_port: right
    output_port: data
"""

# generated from the system
from typing import Dict, Any, List
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    join_type = config.get("join_type", "inner").replace(" ", "_")
    join_condition = config.get("join_conditions", "")
    expressions: List[str] = config.get("expressions", [])

    df_left = inputs.get("left")
    df_right = inputs.get("right")
    if df_left is None or df_right is None:
        raise ValueError("Both left and right inputs must be connected")
    df_left = df_left.alias("left")
    df_right = df_right.alias("right")

    if not join_condition:
        result = df_left.join(df_right, how=join_type)
    else:
        result = df_left.join(df_right, F.expr(join_condition), how=join_type)

    if expressions:
        result = result.selectExpr(*expressions)

    return {"joined_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "left",
    "join_conditions": "left.order_id = right.order_id",
    "expressions": [
        "`left`.order_id",
        "`left`.customer_id",
        "`left`.order_date",
        "`left`.order_status",
        "`right`.order_item_id",
        "`right`.product_id",
        "`right`.quantity",
        "`right`.unit_price",
        "`right`.discount_pct",
        "`right`.line_total"
    ]
}
inputs = {
    "left": ctx["source_0.data"],
    "right": ctx["source_1.data"]
}
out = run(config, inputs, spark)
ctx["join_2.joined_data"] = out["joined_data"]

In [0]:
"""
id: ai_function_15
template: ai_function
templateVersion: 1.0.0
name: sentiment_analysis
position:
  x: 300
  y: 465
description:
  text: Analyze sentiment of review_body and add it as sentiment column.
  hash: e11992ec
previewCodeHash: e15e1c87d75663d0
previewMode: "1000"
config:
  expressions:
    - ai_analyze_sentiment(review_body) AS `sentiment`
input:
  - node: source_14
    input_port: data
    output_port: data
"""

# generated from the system
from typing import Dict, Any, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    expressions: List[str] = config.get("expressions", [])

    if not expressions:
        return {"ai_data": df}

    return {"ai_data": df.selectExpr(*expressions, "*")}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "ai_analyze_sentiment(review_body) AS `sentiment`"
    ]
}
inputs = {
    "data": ctx["source_14.data"]
}
out = run(config, inputs, spark)
ctx["ai_function_15.ai_data"] = out["ai_data"]

In [0]:
"""
id: join_5
template: join
templateVersion: 1.0.0
name: orders_itemsjoinAPI_customer
position:
  x: 600
  y: 155
description:
  text: Left join on customer_id; only keep specified columns from both sides.
  hash: 52ce1eb7
previewCodeHash: 05f273efddd74822
previewMode: "1000"
config:
  join_type: left
  join_conditions: left.customer_id = right.customer_id
  expressions:
    - "`left`.order_id"
    - "`left`.customer_id"
    - "`left`.order_date"
    - "`left`.order_status"
    - "`left`.order_item_id"
    - "`left`.product_id"
    - "`left`.quantity"
    - "`left`.unit_price"
    - "`left`.discount_pct"
    - "`left`.line_total"
    - "`right`.customer_name"
    - "`right`.email"
    - "`right`.phone"
    - "`right`.city"
    - "`right`.state"
    - "`right`.country"
    - "`right`.created_date"
input:
  - node: join_2
    input_port: left
    output_port: joined_data
  - node: python_3
    input_port: right
    output_port: result
"""

# generated from the system
from typing import Dict, Any, List
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    join_type = config.get("join_type", "inner").replace(" ", "_")
    join_condition = config.get("join_conditions", "")
    expressions: List[str] = config.get("expressions", [])

    df_left = inputs.get("left")
    df_right = inputs.get("right")
    if df_left is None or df_right is None:
        raise ValueError("Both left and right inputs must be connected")
    df_left = df_left.alias("left")
    df_right = df_right.alias("right")

    if not join_condition:
        result = df_left.join(df_right, how=join_type)
    else:
        result = df_left.join(df_right, F.expr(join_condition), how=join_type)

    if expressions:
        result = result.selectExpr(*expressions)

    return {"joined_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "left",
    "join_conditions": "left.customer_id = right.customer_id",
    "expressions": [
        "`left`.order_id",
        "`left`.customer_id",
        "`left`.order_date",
        "`left`.order_status",
        "`left`.order_item_id",
        "`left`.product_id",
        "`left`.quantity",
        "`left`.unit_price",
        "`left`.discount_pct",
        "`left`.line_total",
        "`right`.customer_name",
        "`right`.email",
        "`right`.phone",
        "`right`.city",
        "`right`.state",
        "`right`.country",
        "`right`.created_date"
    ]
}
inputs = {
    "left": ctx["join_2.joined_data"],
    "right": ctx["python_3.result"]
}
out = run(config, inputs, spark)
ctx["join_5.joined_data"] = out["joined_data"]

In [0]:
"""
id: workspace.default.output_16
template: output
templateVersion: 1.0.0
name: sentiment
position:
  x: 600
  y: 465
description:
  text: Save data to the specified table, replacing existing data.
  hash: 3c04409e
previewCodeHash: 545374e58c39b5f0
previewMode: "1000"
config:
  catalog: designer_catalog
  schema: enr
  table_name: sentiment
input:
  - node: ai_function_15
    input_port: data
    output_port: ai_data
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    parts = [p for p in [catalog, schema, table_name] if p]
    full_name = ".".join(parts)

    df.write.mode("overwrite").saveAsTable(full_name)

    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "catalog": "designer_catalog",
    "schema": "enr",
    "table_name": "sentiment"
}
inputs = {
    "data": ctx["ai_function_15.ai_data"]
}
out = run(config, inputs, spark)

In [0]:
"""
id: join_6
template: join
templateVersion: 1.0.0
name: OBT
position:
  x: 900
  y: 232.5
description:
  text: Join data on order_id including all records from the left side with selected columns from both sides.
  hash: 5daa293a
previewCodeHash: 2798396235f8eb5e
previewMode: "1000"
config:
  join_type: left
  join_conditions: left.order_id = right.order_id
  expressions:
    - "`left`.order_id"
    - "`left`.customer_id"
    - "`left`.order_date"
    - "`left`.order_status"
    - "`left`.order_item_id"
    - "`left`.product_id"
    - "`left`.quantity"
    - "`left`.unit_price"
    - "`left`.discount_pct"
    - "`left`.line_total"
    - "`left`.customer_name"
    - "`left`.email"
    - "`left`.phone"
    - "`left`.city"
    - "`left`.state"
    - "`left`.country"
    - "`left`.created_date"
    - "`right`.shipment_id"
    - "`right`.ship_date"
    - "`right`.ship_mode"
    - "`right`.shipping_cost"
input:
  - node: join_5
    input_port: left
    output_port: joined_data
  - node: python_4
    input_port: right
    output_port: result
"""

# generated from the system
from typing import Dict, Any, List
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    join_type = config.get("join_type", "inner").replace(" ", "_")
    join_condition = config.get("join_conditions", "")
    expressions: List[str] = config.get("expressions", [])

    df_left = inputs.get("left")
    df_right = inputs.get("right")
    if df_left is None or df_right is None:
        raise ValueError("Both left and right inputs must be connected")
    df_left = df_left.alias("left")
    df_right = df_right.alias("right")

    if not join_condition:
        result = df_left.join(df_right, how=join_type)
    else:
        result = df_left.join(df_right, F.expr(join_condition), how=join_type)

    if expressions:
        result = result.selectExpr(*expressions)

    return {"joined_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "left",
    "join_conditions": "left.order_id = right.order_id",
    "expressions": [
        "`left`.order_id",
        "`left`.customer_id",
        "`left`.order_date",
        "`left`.order_status",
        "`left`.order_item_id",
        "`left`.product_id",
        "`left`.quantity",
        "`left`.unit_price",
        "`left`.discount_pct",
        "`left`.line_total",
        "`left`.customer_name",
        "`left`.email",
        "`left`.phone",
        "`left`.city",
        "`left`.state",
        "`left`.country",
        "`left`.created_date",
        "`right`.shipment_id",
        "`right`.ship_date",
        "`right`.ship_mode",
        "`right`.shipping_cost"
    ]
}
inputs = {
    "left": ctx["join_5.joined_data"],
    "right": ctx["python_4.result"]
}
out = run(config, inputs, spark)
ctx["join_6.joined_data"] = out["joined_data"]

In [0]:
"""
id: aggregate_7
template: aggregate
templateVersion: 1.0.0
name: aggregate_7
position:
  x: 1200
  y: 155
description:
  text: Group data by city and count orders per city. Also include the rounded sum of unit prices.
  hash: 7b75088a
previewCodeHash: 5d1252176836254b
previewMode: "1000"
config:
  group_bys:
    - expr: city
      type: column
  aggregations:
    - columnExpr:
        expr: order_id
        type: column
      fn: COUNT
      alias: TotalOrders
    - columnExpr:
        expr: ROUND(SUM(unit_price),2)
        type: expr
      fn: "-"
      alias: TotalSales
input:
  - node: join_6
    input_port: data
    output_port: joined_data
"""

# generated from the system
import math
from typing import Dict, Any
import pyspark.sql.functions as F

DEFAULT_PERCENTILE = 0.5

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    group_bys = config.get("group_bys", [])
    aggregations = config.get("aggregations", [])

    group_by_set = set(e for gb in group_bys if (e := gb.get("expr", "")))

    agg_exprs = []
    for agg_def in aggregations:
        col_expr = agg_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        fn = agg_def.get("fn", "-")
        alias = agg_def.get("alias")

        if (fn == "-" or fn == "_") and not alias and raw_expr in group_by_set:
            continue

        fn_map = {
            "SUM": F.sum,
            "AVG": F.avg,
            "COUNT": F.count,
            "MIN": F.min,
            "MAX": F.max,
            "MEAN": F.mean,
            "MEDIAN": F.median,
            "STDDEV": F.stddev,
            "VARIANCE": F.variance,
        }

        agg_fn = fn_map.get(fn)
        if agg_fn:
            col = agg_fn(raw_expr)
        elif fn == "-" or fn == "_":
            col = F.expr(raw_expr) if col_expr.get("type") == "expr" else F.col(raw_expr)
        elif fn == "PERCENTILE":
            raw_pct = agg_def.get("percentage")
            if (
                isinstance(raw_pct, (int, float))
                and not isinstance(raw_pct, bool)
                and math.isfinite(raw_pct)
            ):
                pct = max(0.0, min(1.0, float(raw_pct)))
            else:
                pct = DEFAULT_PERCENTILE
            col = F.expr(f"PERCENTILE({raw_expr}, {pct})")
        else:
            col = F.expr(f"{fn}({raw_expr})")

        if alias:
            col = col.alias(alias)

        agg_exprs.append(col)

    group_cols = [
        gb.get("expr", "") for gb in group_bys if gb.get("expr", "")
    ]

    if not agg_exprs:
        if group_cols:
            result = df.select(*group_cols).distinct()
            return {"aggregated_data": result}
        return {"aggregated_data": df}

    if group_cols:
        result = df.groupBy(*group_cols).agg(*agg_exprs)
    else:
        result = df.agg(*agg_exprs)

    return {"aggregated_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "group_bys": [
        {
            "expr": "city",
            "type": "column"
        }
    ],
    "aggregations": [
        {
            "columnExpr": {
                "expr": "order_id",
                "type": "column"
            },
            "fn": "COUNT",
            "alias": "TotalOrders",
            "withAsKeyword": None
        },
        {
            "columnExpr": {
                "expr": "ROUND(SUM(unit_price),2)",
                "type": "expr"
            },
            "fn": "-",
            "alias": "TotalSales",
            "withAsKeyword": None
        }
    ]
}
inputs = {
    "data": ctx["join_6.joined_data"]
}
out = run(config, inputs, spark)
ctx["aggregate_7.aggregated_data"] = out["aggregated_data"]

In [0]:
"""
id: transform_order_month
template: transform
templateVersion: 1.0.0
name: AddOrderMonth
position:
  x: 1200
  y: 310
description:
  text: Add a column extracting the order month from order date.
  hash: 1ca1ecca
previewCodeHash: a64b3b9630b4bd6d
previewMode: "1000"
config:
  expressions:
    - "*"
    - MONTH(order_date) AS `order_month`
input:
  - node: join_6
    input_port: data
    output_port: joined_data
"""

# generated from the system
from typing import Dict, Any, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    expressions: List[str] = config.get("expressions", [])

    if not expressions:
        return {"transformed_data": df}

    return {"transformed_data": df.selectExpr(*expressions)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "*",
        "MONTH(order_date) AS `order_month`"
    ]
}
inputs = {
    "data": ctx["join_6.joined_data"]
}
out = run(config, inputs, spark)
ctx["transform_order_month.transformed_data"] = out["transformed_data"]

In [0]:
"""
id: sort_8
template: sort
templateVersion: 1.0.0
name: sorted
position:
  x: 1500
  y: 155
description:
  text: Sort data by TotalSales in descending order.
  hash: ed5af748
previewCodeHash: 49f3d026aad0388d
previewMode: "1000"
config:
  sort_expressions:
    - columnExpr:
        expr: TotalSales
        type: column
      sortBy: DESC
input:
  - node: aggregate_7
    input_port: data
    output_port: aggregated_data
"""

# generated from the system
from typing import Dict, Any
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    sort_expressions = config.get("sort_expressions", [])

    if not sort_expressions:
        return {"sorted_data": df}

    order_cols = []
    for sort_def in sort_expressions:
        col_expr = sort_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        direction = sort_def.get("sortBy", "UNSET")

        col = F.col(raw_expr)
        if direction == "DESC":
            col = col.desc()
        elif direction == "ASC":
            col = col.asc()

        order_cols.append(col)

    return {"sorted_data": df.orderBy(*order_cols)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "sort_expressions": [
        {
            "columnExpr": {
                "expr": "TotalSales",
                "type": "column"
            },
            "sortBy": "DESC"
        }
    ]
}
inputs = {
    "data": ctx["aggregate_7.aggregated_data"]
}
out = run(config, inputs, spark)
ctx["sort_8.sorted_data"] = out["sorted_data"]

In [0]:
"""
id: aggregate_order_status_per_month
template: aggregate
templateVersion: 1.0.0
name: CountOrderStatusPerMonth
position:
  x: 1500
  y: 310
description:
  text: Group data by order month and status, counting orders in each group.
  hash: 162b04e6
previewCodeHash: a93b8e921c542880
previewMode: "1000"
config:
  group_bys:
    - expr: order_month
      type: expr
    - expr: order_status
      type: expr
  aggregations:
    - columnExpr:
        expr: order_id
        type: expr
      fn: COUNT
      alias: status_count
input:
  - node: transform_order_month
    input_port: data
    output_port: transformed_data
"""

# generated from the system
import math
from typing import Dict, Any
import pyspark.sql.functions as F

DEFAULT_PERCENTILE = 0.5

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    group_bys = config.get("group_bys", [])
    aggregations = config.get("aggregations", [])

    group_by_set = set(e for gb in group_bys if (e := gb.get("expr", "")))

    agg_exprs = []
    for agg_def in aggregations:
        col_expr = agg_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        fn = agg_def.get("fn", "-")
        alias = agg_def.get("alias")

        if (fn == "-" or fn == "_") and not alias and raw_expr in group_by_set:
            continue

        fn_map = {
            "SUM": F.sum,
            "AVG": F.avg,
            "COUNT": F.count,
            "MIN": F.min,
            "MAX": F.max,
            "MEAN": F.mean,
            "MEDIAN": F.median,
            "STDDEV": F.stddev,
            "VARIANCE": F.variance,
        }

        agg_fn = fn_map.get(fn)
        if agg_fn:
            col = agg_fn(raw_expr)
        elif fn == "-" or fn == "_":
            col = F.expr(raw_expr) if col_expr.get("type") == "expr" else F.col(raw_expr)
        elif fn == "PERCENTILE":
            raw_pct = agg_def.get("percentage")
            if (
                isinstance(raw_pct, (int, float))
                and not isinstance(raw_pct, bool)
                and math.isfinite(raw_pct)
            ):
                pct = max(0.0, min(1.0, float(raw_pct)))
            else:
                pct = DEFAULT_PERCENTILE
            col = F.expr(f"PERCENTILE({raw_expr}, {pct})")
        else:
            col = F.expr(f"{fn}({raw_expr})")

        if alias:
            col = col.alias(alias)

        agg_exprs.append(col)

    group_cols = [
        gb.get("expr", "") for gb in group_bys if gb.get("expr", "")
    ]

    if not agg_exprs:
        if group_cols:
            result = df.select(*group_cols).distinct()
            return {"aggregated_data": result}
        return {"aggregated_data": df}

    if group_cols:
        result = df.groupBy(*group_cols).agg(*agg_exprs)
    else:
        result = df.agg(*agg_exprs)

    return {"aggregated_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "group_bys": [
        {
            "expr": "order_month",
            "type": "expr"
        },
        {
            "expr": "order_status",
            "type": "expr"
        }
    ],
    "aggregations": [
        {
            "columnExpr": {
                "expr": "order_id",
                "type": "expr"
            },
            "fn": "COUNT",
            "alias": "status_count",
            "withAsKeyword": None
        }
    ]
}
inputs = {
    "data": ctx["transform_order_month.transformed_data"]
}
out = run(config, inputs, spark)
ctx["aggregate_order_status_per_month.aggregated_data"] = out["aggregated_data"]

In [0]:
"""
id: workspace.default.output_9
template: output
templateVersion: 1.0.0
name: AggregatedOrders
position:
  x: 1800
  y: 155
description:
  text: Save data by overwriting a specific table.
  hash: 6b6437d4
previewCodeHash: 011cbdbca473f033
previewMode: "1000"
config:
  catalog: designer_catalog
  schema: enr
  table_name: AggregatedOrders
input:
  - node: sort_8
    input_port: data
    output_port: sorted_data
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    parts = [p for p in [catalog, schema, table_name] if p]
    full_name = ".".join(parts)

    df.write.mode("overwrite").saveAsTable(full_name)

    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "catalog": "designer_catalog",
    "schema": "enr",
    "table_name": "AggregatedOrders"
}
inputs = {
    "data": ctx["sort_8.sorted_data"]
}
out = run(config, inputs, spark)

In [0]:
"""
id: sort_order_status_per_month
template: sort
templateVersion: 1.0.0
name: SortOrderStatusPerMonth
position:
  x: 1800
  y: 310
description:
  text: Sort data by order_month in descending order.
  hash: 40d81d3c
previewCodeHash: 9d8892300448e458
previewMode: "1000"
config:
  sort_expressions:
    - columnExpr:
        expr: order_month
        type: expr
      sortBy: DESC
input:
  - node: aggregate_order_status_per_month
    input_port: data
    output_port: aggregated_data
"""

# generated from the system
from typing import Dict, Any
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    sort_expressions = config.get("sort_expressions", [])

    if not sort_expressions:
        return {"sorted_data": df}

    order_cols = []
    for sort_def in sort_expressions:
        col_expr = sort_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        direction = sort_def.get("sortBy", "UNSET")

        col = F.col(raw_expr)
        if direction == "DESC":
            col = col.desc()
        elif direction == "ASC":
            col = col.asc()

        order_cols.append(col)

    return {"sorted_data": df.orderBy(*order_cols)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "sort_expressions": [
        {
            "columnExpr": {
                "expr": "order_month",
                "type": "expr"
            },
            "sortBy": "DESC"
        }
    ]
}
inputs = {
    "data": ctx["aggregate_order_status_per_month.aggregated_data"]
}
out = run(config, inputs, spark)
ctx["sort_order_status_per_month.sorted_data"] = out["sorted_data"]

In [0]:
"""
id: workspace.default.output_13
template: output
templateVersion: 1.0.0
name: OrderStatus
position:
  x: 2100
  y: 310
description:
  text: Save data by replacing existing table in specified catalog and schema.
  hash: e869fd66
previewMode: "1000"
config:
  catalog: designer_catalog
  schema: enr
  table_name: OrderStatus
input:
  - node: sort_order_status_per_month
    input_port: data
    output_port: sorted_data
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    parts = [p for p in [catalog, schema, table_name] if p]
    full_name = ".".join(parts)

    df.write.mode("overwrite").saveAsTable(full_name)

    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "catalog": "designer_catalog",
    "schema": "enr",
    "table_name": "OrderStatus"
}
inputs = {
    "data": ctx["sort_order_status_per_month.sorted_data"]
}
out = run(config, inputs, spark)